In [1]:
# =========== IMPORT LIBRARY & SETUP ==========
import numpy as np
import pandas as pd
import random
import plotly.graph_objects as go
import plotly.colors as pc
import os
import time
import warnings
from collections import defaultdict

warnings.filterwarnings('ignore')

# =========== NUMBA SETUP ==========
try:
    from numba import jit, njit, prange
    import numba as nb
    NUMBA_AVAILABLE = True
    print("✅ Numba available - JIT compilation ENABLED")
    
    import multiprocessing
    cpu_cores = multiprocessing.cpu_count()
    print(f"   CPU Cores detected: {cpu_cores}")
    print(f"   Parallel mode: ENABLED (prange with {cpu_cores} threads)")
    
except ImportError:
    NUMBA_AVAILABLE = False
    print("⚠️ Numba not available - install with: pip install numba")

print("="*60)
print("3D BIN PACKING - NUMBA PARALLEL OPTIMIZED")
print("="*60)

✅ Numba available - JIT compilation ENABLED
   CPU Cores detected: 16
   Parallel mode: ENABLED (prange with 16 threads)
3D BIN PACKING - NUMBA PARALLEL OPTIMIZED


In [ ]:
# =========== KONFIGURASI DATA ==========
class Container:
    def __init__(self, length=240.0, width=160.0, height=130.0, max_weight=800.0):
        self.length = float(length)
        self.width = float(width)
        self.height = float(height)
        self.max_weight = float(max_weight)
        self.volume = self.length * self.width * self.height
        self.cog_limit_x = length * 0.1
        self.cog_limit_y = width * 0.1
        self.cog_limit_z = height * 0.15

class Package:
    def __init__(self, id_, length, width, height, weight):
        self.id = id_
        self.original = (float(length), float(width), float(height))
        self.weight = float(weight)
        self.volume = float(length) * float(width) * float(height)
        self.orientations = self.generate_orientations()

    def generate_orientations(self):
        l, w, h = self.original
        return {
            1: (l, w, h), 2: (l, h, w), 3: (w, l, h),
            4: (w, h, l), 5: (h, l, w), 6: (h, w, l)
        }
    
# =========== NUMBA DATA INITIALIZATION ==========
_numba_package_dims = None
_numba_package_weights = None
_numba_package_id_to_idx = None
_numba_initialized = False

def initialize_numba_data(packages_dict):
    global _numba_package_dims, _numba_package_weights, _numba_package_id_to_idx, _numba_initialized
    
    if not NUMBA_AVAILABLE:
        return False
    
    package_ids = list(packages_dict.keys())
    _numba_package_id_to_idx = {pid: i for i, pid in enumerate(package_ids)}
    
    n_packages = len(package_ids)
    _numba_package_dims = np.zeros((n_packages, 18), dtype=np.float32)
    _numba_package_weights = np.zeros(n_packages, dtype=np.float32)
    
    for pid, pkg in packages_dict.items():
        idx = _numba_package_id_to_idx[pid]
        _numba_package_weights[idx] = pkg.weight
        for rot in range(1, 7):
            dims = pkg.orientations[rot]
            _numba_package_dims[idx, (rot-1)*3] = dims[0]
            _numba_package_dims[idx, (rot-1)*3 + 1] = dims[1]
            _numba_package_dims[idx, (rot-1)*3 + 2] = dims[2]
    
    _numba_initialized = True
    print(f"✅ Numba data initialized for {n_packages} packages")
    return True

# =========== OPTIMIZED BLF ==========
@njit(cache=True, fastmath=True)
def bottom_left_fill_numba_optimized(chromosome_ids, chromosome_rots, 
                                      package_dims_array, package_weights_array,
                                      container_len, container_wid, container_hei,
                                      container_cog_limit_x, container_cog_limit_y, container_cog_limit_z,
                                      container_max_weight):
    """
    Menerima: 1 kromosom (array of package IDs + rotations)
    Mengembalikan: posisi, fitness, dll untuk 1 kromosom tersebut
    
    MEKANISME BLF DI SINI:
    1. Inisialisasi grid, skyline, support_map
    2. Untuk setiap paket dalam kromosom (berurutan):
       a. Scan semua (x,y) cari Z minimal global
       b. Pilih posisi dengan Z terkecil (→ Y terkecil → X terkecil)
       c. Tempatkan paket
       d. Update skyline dan support_map
    3. Hitung fitness (volume - penalty, dikalikan B4*B5)
    """
    L, W, H = int(container_len), int(container_wid), int(container_hei)
    L_int, W_int, H_int = L + 1, W + 1, H + 1
    
    # Grid untuk collision check
    grid = np.zeros((L_int, W_int, H_int), dtype=np.uint8)
    
    # SKYLINE: permukaan tertinggi yang terisi per (x,y)
    skyline = np.zeros((L_int, W_int), dtype=np.int16)
    
    # SUPPORT MAP: jumlah paket penopang per (x,y) untuk stability check
    support_map = np.zeros((L_int, W_int), dtype=np.int16)
    
    n_packages = len(chromosome_ids)
    
    # Arrays untuk posisi
    pos_x = np.zeros(n_packages, dtype=np.float32)
    pos_y = np.zeros(n_packages, dtype=np.float32)
    pos_z = np.zeros(n_packages, dtype=np.float32)
    placed_flags = np.zeros(n_packages, dtype=np.int8)
    package_volumes = np.zeros(n_packages, dtype=np.float32)
    
    total_volume = 0.0
    total_weight = 0.0
    total_mass_x = 0.0
    total_mass_y = 0.0
    total_mass_z = 0.0
    all_stability_valid = True
    num_placed = 0
    
    for idx in range(n_packages):
        p_id = chromosome_ids[idx]
        rot = chromosome_rots[idx]
        
        base_idx = rot * 3
        dx = package_dims_array[p_id, base_idx]
        dy = package_dims_array[p_id, base_idx + 1]
        dz = package_dims_array[p_id, base_idx + 2]
        
        dx_int = int(np.ceil(dx))
        dy_int = int(np.ceil(dy))
        dz_int = int(np.ceil(dz))
        
        max_x = L - dx_int + 1
        max_y = W - dy_int + 1
        
        placed = False
        best_x, best_y, best_z = -1, -1, -1
        
        # ========== Z-PRIORITY SCAN DENGAN EARLY EXIT ==========
        # Inisialisasi
        min_z_global = 999999
        best_y_candidate = 999999
        best_x_candidate = 999999
        
        # Flag untuk early exit jika sudah nemu Z=0
        found_zero_z = False
        
        # SCAN Y terluar, X terdalam
        for y in range(max_y):
            if found_zero_z:
                break
            
            for x in range(max_x):
                # ========== HITUNG Z_MIN DARI SKYLINE ==========
                # Minimalisir loop dengan break awal jika sudah Z=0
                z_min = 0
                for i in range(dx_int):
                    for j in range(dy_int):
                        val = skyline[x + i, y + j]
                        if val > z_min:
                            z_min = val
                            # Early exit dalam loop: jika sudah >= min_z_global, 
                            # posisi ini tidak akan lebih baik
                            if z_min >= min_z_global:
                                break
                    if z_min >= min_z_global:
                        break
                
                # Jika z_min sudah lebih besar dari min_z_global, skip
                if z_min >= min_z_global and min_z_global != 999999:
                    continue
                
                # Cek apakah muat di ketinggian ini
                if z_min + dz_int > H:
                    continue
                
                # ========== COLLISION CHECK DI Z_MIN ==========
                collision = False
                for i in range(dx_int):
                    for j in range(dy_int):
                        for k in range(dz_int):
                            if grid[x + i, y + j, z_min + k] != 0:
                                collision = True
                                break
                        if collision:
                            break
                    if collision:
                        break
                
                if collision:
                    # Jika collision, cari Z yang lebih tinggi
                    # Optimasi: langsung hitung max occupied Z + 1
                    max_z_in_area = 0
                    for i in range(dx_int):
                        for j in range(dy_int):
                            for k in range(dz_int):
                                if grid[x + i, y + j, z_min + k] != 0:
                                    found_z = z_min + k + 1
                                    if found_z > max_z_in_area:
                                        max_z_in_area = found_z
                    z_candidate = max_z_in_area
                    
                    if z_candidate + dz_int > H:
                        continue
                    
                    # Re-check collision
                    collision = False
                    for i in range(dx_int):
                        for j in range(dy_int):
                            for k in range(dz_int):
                                if grid[x + i, y + j, z_candidate + k] != 0:
                                    collision = True
                                    break
                            if collision:
                                break
                        if collision:
                            break
                    
                    if collision:
                        continue
                else:
                    z_candidate = z_min
                
                # ========== CEK STABILITAS (B5) ==========
                stability_valid = True
                if z_candidate > 0:
                    support_count = 0
                    for i in range(dx_int):
                        for j in range(dy_int):
                            if support_map[x + i, y + j] > 0:
                                support_count += 1
                    
                    if support_count < (dx_int * dy_int) * 0.5:
                        stability_valid = False
                        all_stability_valid = False
                
                if not stability_valid:
                    continue
                
                # ========== PRIORITAS: Z TERKECIL ==========
                if z_candidate < min_z_global:
                    min_z_global = z_candidate
                    best_y_candidate = y
                    best_x_candidate = x
                    
                    # ========== EARLY EXIT: jika Z=0, langsung stop ==========
                    if min_z_global == 0:
                        found_zero_z = True
                        break
                
                # Jika Z sama, pilih Y terkecil
                elif z_candidate == min_z_global:
                    if y < best_y_candidate:
                        best_y_candidate = y
                        best_x_candidate = x
                    # Jika Y sama, pilih X terkecil
                    elif y == best_y_candidate and x < best_x_candidate:
                        best_x_candidate = x
        
        # Setelah scan selesai, cek apakah ada posisi yang valid
        if min_z_global != 999999:
            best_x = best_x_candidate
            best_y = best_y_candidate
            best_z = min_z_global
            placed = True
            
            # ========== TEMPATKAN PAKET ==========
            # Update grid
            for i in range(dx_int):
                for j in range(dy_int):
                    for k in range(dz_int):
                        grid[best_x + i, best_y + j, best_z + k] = p_id + 1
            
            # Update skyline (permukaan tertinggi)
            new_top = best_z + dz_int
            for i in range(dx_int):
                for j in range(dy_int):
                    if new_top > skyline[best_x + i, best_y + j]:
                        skyline[best_x + i, best_y + j] = new_top
            
            # Update support map (untuk paket di atas)
            for i in range(dx_int):
                for j in range(dy_int):
                    support_map[best_x + i, best_y + j] += 1
            
            # Hitung statistik
            volume = dx * dy * dz
            weight = package_weights_array[p_id]
            cog_x = best_x + dx / 2.0
            cog_y = best_y + dy / 2.0
            cog_z = best_z + dz / 2.0
            
            total_volume += volume
            total_weight += weight
            total_mass_x += weight * cog_x
            total_mass_y += weight * cog_y
            total_mass_z += weight * cog_z
            num_placed += 1
            
            pos_x[idx] = best_x
            pos_y[idx] = best_y
            pos_z[idx] = best_z
            placed_flags[idx] = 1
            package_volumes[idx] = volume
        
        if not placed:
            placed_flags[idx] = 0
            package_volumes[idx] = dx * dy * dz
            pos_x[idx] = -1
            pos_y[idx] = -1
            pos_z[idx] = -1
    
    # Hitung pusat massa total
    if total_weight > 0:
        cog_total_x = total_mass_x / total_weight
        cog_total_y = total_mass_y / total_weight
        cog_total_z = total_mass_z / total_weight
    else:
        cog_total_x = container_len / 2.0
        cog_total_y = container_wid / 2.0
        cog_total_z = container_hei / 2.0
    
    # Hitung deviasi dari pusat kontainer (B1, B2, B3)
    container_center_x = container_len / 2.0
    container_center_y = container_wid / 2.0
    container_center_z = container_hei / 2.0
    
    dev_x = abs(cog_total_x - container_center_x)
    dev_y = abs(cog_total_y - container_center_y)
    dev_z = abs(cog_total_z - container_center_z)
    
    B1 = max(0.0, dev_x - container_cog_limit_x)
    B2 = max(0.0, dev_y - container_cog_limit_y)
    B3 = max(0.0, dev_z - container_cog_limit_z)
    
    penalty_cog = B1 + B2 + B3
    B4 = 1 if total_weight <= container_max_weight else 0
    B5 = 1 if all_stability_valid else 0
    
    fitness_raw = total_volume - penalty_cog
    fitness_final = fitness_raw * B4 * B5
    
    return (fitness_final, total_volume, total_weight, penalty_cog, 
            B1, B2, B3, B4, B5, num_placed,
            cog_total_x, cog_total_y, cog_total_z, dev_x, dev_y, dev_z,
            pos_x, pos_y, pos_z, placed_flags, package_volumes)


# =========== PARALLEL BATCH EVALUATION (OPTIMIZED) ==========
@njit(parallel=True, cache=True, fastmath=True)
def evaluate_population_parallel_numba(chromosomes_ids_batch, chromosomes_rots_batch,
                                         package_dims_array, package_weights_array,
                                         container_len, container_wid, container_hei,
                                         container_cog_limit_x, container_cog_limit_y, container_cog_limit_z,
                                         container_max_weight):
    """
    Menerima: BANYAK kromosom (2D array: [n_chrom x n_packages])
    Mengembalikan: fitness untuk semua kromosom
    
    MEKANISME DI SINI:
    for c in prange(n_chrom):  # PARALLEL untuk setiap core CPU
        # ============================================
        # KODE DI BAWAH INI ADALAH DUPLIKASI DARI
        # bottom_left_fill_numba_optimized() !!!
        # ============================================
        # Inisialisasi grid, skyline, support_map untuk kromosom c
        # Untuk setiap paket dalam kromosom c:
        #    Scan (x,y) cari Z minimal
        #    Place package, update skyline
        # Hitung fitness untuk kromosom c
        # ============================================
    """
    n_chrom = chromosomes_ids_batch.shape[0]
    n_packages = chromosomes_ids_batch.shape[1]
    
    # Array untuk menyimpan hasil
    all_fitness = np.zeros(n_chrom, dtype=np.float64)
    all_total_volume = np.zeros(n_chrom, dtype=np.float64)
    all_total_weight = np.zeros(n_chrom, dtype=np.float64)
    all_penalty_cog = np.zeros(n_chrom, dtype=np.float64)
    all_B1 = np.zeros(n_chrom, dtype=np.float64)
    all_B2 = np.zeros(n_chrom, dtype=np.float64)
    all_B3 = np.zeros(n_chrom, dtype=np.float64)
    all_B4 = np.zeros(n_chrom, dtype=np.int8)
    all_B5 = np.zeros(n_chrom, dtype=np.int8)
    all_num_placed = np.zeros(n_chrom, dtype=np.int32)
    all_cog_x = np.zeros(n_chrom, dtype=np.float64)
    all_cog_y = np.zeros(n_chrom, dtype=np.float64)
    all_cog_z = np.zeros(n_chrom, dtype=np.float64)
    
    for c in prange(n_chrom):
        chrom_ids = chromosomes_ids_batch[c]
        chrom_rots = chromosomes_rots_batch[c]
        
        L, W, H = int(container_len), int(container_wid), int(container_hei)
        L_int, W_int, H_int = L + 1, W + 1, H + 1
        
        # Grid
        grid = np.zeros((L_int, W_int, H_int), dtype=np.uint8)
        
        # Skyline
        skyline = np.zeros((L_int, W_int), dtype=np.int16)
        
        # Support map
        support_map = np.zeros((L_int, W_int), dtype=np.int16)
        
        total_volume = 0.0
        total_weight = 0.0
        total_mass_x = 0.0
        total_mass_y = 0.0
        total_mass_z = 0.0
        all_stability_valid = True
        num_placed = 0
        
        for idx in range(n_packages):
            p_id = chrom_ids[idx]
            rot = chrom_rots[idx]
            
            base_idx = rot * 3
            dx = package_dims_array[p_id, base_idx]
            dy = package_dims_array[p_id, base_idx + 1]
            dz = package_dims_array[p_id, base_idx + 2]
            
            dx_int = int(np.ceil(dx))
            dy_int = int(np.ceil(dy))
            dz_int = int(np.ceil(dz))
            
            max_x = L - dx_int + 1
            max_y = W - dy_int + 1
            
            placed = False
            best_x, best_y, best_z = -1, -1, -1
            
            # ========== Z-PRIORITY SCAN DENGAN EARLY EXIT ==========
            min_z_global = 999999
            best_y_candidate = 999999
            best_x_candidate = 999999
            found_zero_z = False
            
            for y in range(max_y):
                if found_zero_z:
                    break
                
                for x in range(max_x):
                    # Hitung Z minimal dari skyline
                    z_min = 0
                    for i in range(dx_int):
                        for j in range(dy_int):
                            val = skyline[x + i, y + j]
                            if val > z_min:
                                z_min = val
                                if z_min >= min_z_global:
                                    break
                        if z_min >= min_z_global:
                            break
                    
                    if z_min >= min_z_global and min_z_global != 999999:
                        continue
                    
                    if z_min + dz_int > H:
                        continue
                    
                    # Collision check
                    collision = False
                    for i in range(dx_int):
                        for j in range(dy_int):
                            for k in range(dz_int):
                                if grid[x + i, y + j, z_min + k] != 0:
                                    collision = True
                                    break
                            if collision:
                                break
                        if collision:
                            break
                    
                    if collision:
                        # Cari z yang lebih tinggi
                        max_z_in_area = 0
                        for i in range(dx_int):
                            for j in range(dy_int):
                                for k in range(dz_int):
                                    if grid[x + i, y + j, z_min + k] != 0:
                                        found_z = z_min + k + 1
                                        if found_z > max_z_in_area:
                                            max_z_in_area = found_z
                        z_candidate = max_z_in_area
                        
                        if z_candidate + dz_int > H:
                            continue
                        
                        # Re-check collision
                        collision = False
                        for i in range(dx_int):
                            for j in range(dy_int):
                                for k in range(dz_int):
                                    if grid[x + i, y + j, z_candidate + k] != 0:
                                        collision = True
                                        break
                                if collision:
                                    break
                            if collision:
                                break
                        
                        if collision:
                            continue
                    else:
                        z_candidate = z_min
                    
                    # Stability check
                    stability_valid = True
                    if z_candidate > 0:
                        support_count = 0
                        for i in range(dx_int):
                            for j in range(dy_int):
                                if support_map[x + i, y + j] > 0:
                                    support_count += 1
                        
                        if support_count < (dx_int * dy_int) * 0.5:
                            stability_valid = False
                            all_stability_valid = False
                    
                    if not stability_valid:
                        continue
                    
                    # Z-priority selection
                    if z_candidate < min_z_global:
                        min_z_global = z_candidate
                        best_y_candidate = y
                        best_x_candidate = x
                        
                        # EARLY EXIT: jika Z=0, langsung stop
                        if min_z_global == 0:
                            found_zero_z = True
                            break
                    
                    elif z_candidate == min_z_global:
                        if y < best_y_candidate:
                            best_y_candidate = y
                            best_x_candidate = x
                        elif y == best_y_candidate and x < best_x_candidate:
                            best_x_candidate = x
            
            if min_z_global != 999999:
                best_x = best_x_candidate
                best_y = best_y_candidate
                best_z = min_z_global
                placed = True
                
                # Place package
                for i in range(dx_int):
                    for j in range(dy_int):
                        for k in range(dz_int):
                            grid[best_x + i, best_y + j, best_z + k] = p_id + 1
                
                # Update skyline
                new_top = best_z + dz_int
                for i in range(dx_int):
                    for j in range(dy_int):
                        if new_top > skyline[best_x + i, best_y + j]:
                            skyline[best_x + i, best_y + j] = new_top
                
                # Update support map
                for i in range(dx_int):
                    for j in range(dy_int):
                        support_map[best_x + i, best_y + j] += 1
                
                volume = dx * dy * dz
                weight = package_weights_array[p_id]
                cog_x = best_x + dx / 2.0
                cog_y = best_y + dy / 2.0
                cog_z = best_z + dz / 2.0
                
                total_volume += volume
                total_weight += weight
                total_mass_x += weight * cog_x
                total_mass_y += weight * cog_y
                total_mass_z += weight * cog_z
                num_placed += 1
        
        # Hitung fitness
        if total_weight > 0:
            cog_total_x = total_mass_x / total_weight
            cog_total_y = total_mass_y / total_weight
            cog_total_z = total_mass_z / total_weight
        else:
            cog_total_x = container_len / 2.0
            cog_total_y = container_wid / 2.0
            cog_total_z = container_hei / 2.0
        
        container_center_x = container_len / 2.0
        container_center_y = container_wid / 2.0
        container_center_z = container_hei / 2.0
        
        dev_x = abs(cog_total_x - container_center_x)
        dev_y = abs(cog_total_y - container_center_y)
        dev_z = abs(cog_total_z - container_center_z)
        
        B1 = max(0.0, dev_x - container_cog_limit_x)
        B2 = max(0.0, dev_y - container_cog_limit_y)
        B3 = max(0.0, dev_z - container_cog_limit_z)
        
        penalty_cog = B1 + B2 + B3
        B4 = 1 if total_weight <= container_max_weight else 0
        B5 = 1 if all_stability_valid else 0
        
        fitness_raw = total_volume - penalty_cog
        fitness_final = fitness_raw * B4 * B5
        
        all_fitness[c] = fitness_final
        all_total_volume[c] = total_volume
        all_total_weight[c] = total_weight
        all_penalty_cog[c] = penalty_cog
        all_B1[c] = B1
        all_B2[c] = B2
        all_B3[c] = B3
        all_B4[c] = B4
        all_B5[c] = B5
        all_num_placed[c] = num_placed
        all_cog_x[c] = cog_total_x
        all_cog_y[c] = cog_total_y
        all_cog_z[c] = cog_total_z
    
    return (all_fitness, all_total_volume, all_total_weight, all_penalty_cog,
            all_B1, all_B2, all_B3, all_B4, all_B5, all_num_placed,
            all_cog_x, all_cog_y, all_cog_z)

# =========== WRAPPER EVALUASI POPULASI ==========
def evaluate_population_parallel(population, container, packages_dict):
    global _numba_initialized
    
    if not NUMBA_AVAILABLE:
        return [], []
    
    if not _numba_initialized:
        initialize_numba_data(packages_dict)
    
    n_chrom = len(population)
    n_packages = len(population[0]) if n_chrom > 0 else 0
    
    # Siapkan batch array
    chrom_ids_batch = np.zeros((n_chrom, n_packages), dtype=np.int32)
    chrom_rots_batch = np.zeros((n_chrom, n_packages), dtype=np.int32)
    
    for i, chrom in enumerate(population):
        for j, (pid, rot) in enumerate(chrom):
            chrom_ids_batch[i, j] = _numba_package_id_to_idx[pid]
            chrom_rots_batch[i, j] = rot - 1
    
    # Panggil parallel evaluation dengan optimized BLF
    (all_fitness, all_total_volume, all_total_weight, all_penalty_cog,
     all_B1, all_B2, all_B3, all_B4, all_B5, all_num_placed,
     all_cog_x, all_cog_y, all_cog_z) = evaluate_population_parallel_numba(
        chrom_ids_batch, chrom_rots_batch,
        _numba_package_dims, _numba_package_weights,
        container.length, container.width, container.height,
        container.cog_limit_x, container.cog_limit_y, container.cog_limit_z,
        container.max_weight
    )
    
    # Build metadata
    fitness_scores = []
    metadata_list = []
    container_volume = container.volume
    
    for i, chrom in enumerate(population):
        fitness = float(all_fitness[i])
        total_volume = float(all_total_volume[i])
        total_weight = float(all_total_weight[i])
        penalty_cog = float(all_penalty_cog[i])
        B1 = float(all_B1[i])
        B2 = float(all_B2[i])
        B3 = float(all_B3[i])
        B4 = int(all_B4[i])
        B5 = int(all_B5[i])
        num_placed = int(all_num_placed[i])
        cog_x = float(all_cog_x[i])
        cog_y = float(all_cog_y[i])
        cog_z = float(all_cog_z[i])
        
        volume_utilization = (total_volume / container_volume) * 100 if container_volume > 0 else 0
        
        # Simplified positions (untuk metadata)
        positions = []
        for j, (pid, rot) in enumerate(chrom):
            dims = packages_dict[pid].orientations[rot]
            positions.append({
                'id': pid,
                'x': -1, 'y': -1, 'z': -1,
                'dx': dims[0], 'dy': dims[1], 'dz': dims[2],
                'weight': packages_dict[pid].weight,
                'volume': dims[0] * dims[1] * dims[2],
                'orientation': rot,
                'placed': j < num_placed
            })
        
        metadata = {
            'fitness': fitness,
            'volume_utilization': volume_utilization,
            'total_volume': total_volume,
            'total_weight': total_weight,
            'penalty_cog': penalty_cog,
            'B1': B1, 'B2': B2, 'B3': B3,
            'B4': B4, 'B5': B5,
            'num_placed': num_placed,
            'positions': positions,
            'cog_actual': (cog_x, cog_y, cog_z),
            'cog_deviation': (abs(cog_x - container.length/2), 
                              abs(cog_y - container.width/2), 
                              abs(cog_z - container.height/2))
        }
        
        fitness_scores.append(fitness)
        metadata_list.append(metadata)
    
    return fitness_scores, metadata_list

# =========== FUNGSI GENETIC ALGORITHM ==========
def create_chromosome(packages_list):
    ids = [p.id for p in packages_list]
    random.shuffle(ids)
    chromosome = []
    for p_id in ids:
        orientation = random.randint(1, 6)
        chromosome.append((p_id, orientation))
    return chromosome

def tournament_selection(population, fitness_scores, tournament_size=3):
    indices = random.sample(range(len(population)), tournament_size)
    best_idx = max(indices, key=lambda i: fitness_scores[i])
    return best_idx

def pmx_crossover(parent1, parent2):
    size = len(parent1)
    if size < 2:
        return parent1.copy()
    
    p1 = parent1.copy()
    p2 = parent2.copy()
    
    cut1 = random.randint(0, size - 2)
    cut2 = random.randint(cut1 + 1, size - 1)
    
    child = [None] * size
    child[cut1:cut2] = p1[cut1:cut2]
    
    mapping = {}
    for i in range(cut1, cut2):
        mapping[p1[i][0]] = p2[i][0]
    
    for i in range(size):
        if i < cut1 or i >= cut2:
            gene = p2[i]
            used_ids = {g[0] for g in child if g is not None}
            while gene[0] in used_ids:
                if gene[0] in mapping:
                    mapped_id = mapping[gene[0]]
                    for g in p2:
                        if g[0] == mapped_id:
                            gene = g
                            break
                else:
                    for g in p1:
                        if g[0] not in used_ids:
                            gene = g
                            break
            child[i] = gene
    
    return child

def mutate(chromosome, mutation_rate=0.1):
    if random.random() > mutation_rate:
        return chromosome.copy()
    
    mutated = chromosome.copy()
    if len(mutated) == 0:
        return mutated
    
    mutation_type = random.choice(['swap_order', 'swap_rotation'])
    
    if mutation_type == 'swap_order':
        if len(mutated) >= 2:
            idx1, idx2 = random.sample(range(len(mutated)), 2)
            mutated[idx1], mutated[idx2] = mutated[idx2], mutated[idx1]
    else:
        idx = random.randint(0, len(mutated)-1)
        old_orient = mutated[idx][1]
        new_orient = random.randint(1, 6)
        while new_orient == old_orient:
            new_orient = random.randint(1, 6)
        mutated[idx] = (mutated[idx][0], new_orient)
    
    return mutated

# =========== VISUALISASI 3D ==========
def visualize_packing(positions, container_dims, title="3D Bin Packing"):
    fig = go.Figure()
    
    container_lines = [
        [0, 1], [1, 2], [2, 3], [3, 0],
        [4, 5], [5, 6], [6, 7], [7, 4],
        [0, 4], [1, 5], [2, 6], [3, 7]
    ]
    
    container_vertices = [
        [0, 0, 0], [container_dims[0], 0, 0],
        [container_dims[0], container_dims[1], 0], [0, container_dims[1], 0],
        [0, 0, container_dims[2]], [container_dims[0], 0, container_dims[2]],
        [container_dims[0], container_dims[1], container_dims[2]],
        [0, container_dims[1], container_dims[2]]
    ]
    
    for i, line in enumerate(container_lines):
        fig.add_trace(go.Scatter3d(
            x=[container_vertices[line[0]][0], container_vertices[line[1]][0]],
            y=[container_vertices[line[0]][1], container_vertices[line[1]][1]],
            z=[container_vertices[line[0]][2], container_vertices[line[1]][2]],
            mode='lines',
            line=dict(color='gray', width=2, dash='dash'),
            name='Container',
            showlegend=(i == 0)
        ))
    
    colors = pc.qualitative.Plotly + pc.qualitative.Dark24 + pc.qualitative.Light24
    placed_positions = [p for p in positions if p.get('placed', False)]
    valid_positions = [p for p in placed_positions if p['x'] >= 0]
    
    for i, pos in enumerate(valid_positions):
        x, y, z = pos['x'], pos['y'], pos['z']
        dx, dy, dz = pos['dx'], pos['dy'], pos['dz']
        color = colors[i % len(colors)]
        
        # Tambahkan edge line untuk setiap balok agar lebih jelas
        fig.add_trace(go.Mesh3d(
            x=[x, x+dx, x+dx, x, x, x+dx, x+dx, x],
            y=[y, y, y+dy, y+dy, y, y, y+dy, y+dy],
            z=[z, z, z, z, z+dz, z+dz, z+dz, z+dz],
            i=[0, 0, 0, 1, 1, 2],
            j=[1, 2, 4, 3, 5, 6],
            k=[2, 3, 5, 7, 6, 7],
            color=color,
            opacity=0.85,           # Meningkatkan opacity untuk tampilan lebih solid
            flatshading=True,
            lighting=dict(ambient=0.5, diffuse=0.8, roughness=0.5),
            name=f"{pos['id']}"
        ))
    
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center'),
        scene=dict(
            xaxis_title='Panjang (X) cm',
            yaxis_title='Lebar (Y) cm',
            zaxis_title='Tinggi (Z) cm',
            aspectmode='data',
            camera=dict(up=dict(x=0, y=0, z=1), eye=dict(x=1.5, y=1.5, z=1.5))
        ),
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    return fig

# =========== MAIN PROGRAM ==========
def load_packages_from_csv(file_path):
    df = pd.read_csv(file_path)
    packages = []
    for _, row in df.iterrows():
        package = Package(
            id_=str(row['id']),
            length=float(row['length_cm']),
            width=float(row['width_cm']),
            height=float(row['height_cm']),
            weight=float(row['weight_kg'])
        )
        packages.append(package)
    return packages

def main():
    print("\n>>> MEMBACA FILE dataset.csv <<<\n")
    
    if not os.path.exists("dataset.csv"):
        print("ERROR: File dataset.csv tidak ditemukan!")
        return
    
    packages = load_packages_from_csv("dataset.csv")
    print(f"Berhasil memuat {len(packages)} paket dari dataset.csv")
    
    container = Container()
    packages_dict = {p.id: p for p in packages}
    
    print(f"\n=== KONTAINER ===")
    print(f"Dimensi: {container.length} x {container.width} x {container.height} cm")
    print(f"Volume: {container.volume} cm³")
    print(f"Berat maks: {container.max_weight} kg")
    print(f"Batas pusat massa: X={container.cog_limit_x:.1f}, Y={container.cog_limit_y:.1f}, Z={container.cog_limit_z:.1f} cm")
    
    total_volume_all = sum(p.volume for p in packages)
    total_weight_all = sum(p.weight for p in packages)
    
    print(f"\nTotal volume semua paket: {total_volume_all:.0f} cm³")
    print(f"Total berat semua paket: {total_weight_all:.1f} kg")
    print(f"Volume kontainer: {container.volume} cm³")
    print(f"Utilization maksimum teoritis: {100 * total_volume_all / container.volume:.1f}%")
    
    if NUMBA_AVAILABLE:
        initialize_numba_data(packages_dict)
    
    print("\n=== KONFIGURASI GENETIC ALGORITHM ===\n")
    
    while True:
        try:
            population_size = int(input("Masukkan jumlah populasi (contoh: 50): "))
            if population_size > 0:
                break
            print("Populasi harus > 0!")
        except ValueError:
            print("Masukkan angka!")
    
    while True:
        try:
            generations = int(input("Masukkan jumlah generasi (contoh: 50): "))
            if generations > 0:
                break
            print("Generasi harus > 0!")
        except ValueError:
            print("Masukkan angka!")
    
    while True:
        try:
            crossover_rate = float(input("Masukkan crossover rate (0-1, contoh: 0.8): "))
            if 0 <= crossover_rate <= 1:
                break
            print("Crossover rate harus 0-1!")
        except ValueError:
            print("Masukkan angka!")
    
    while True:
        try:
            mutation_rate = float(input("Masukkan mutation rate (0-1, contoh: 0.2): "))
            if 0 <= mutation_rate <= 1:
                break
            print("Mutation rate harus 0-1!")
        except ValueError:
            print("Masukkan angka!")
    
    import multiprocessing
    cpu_cores = multiprocessing.cpu_count()
    
    # for testing in range(2):
    print(f"\nKonfigurasi GA:")
    print(f"  - Jumlah Paket: {len(packages)}")
    print(f"  - Populasi: {population_size}")
    print(f"  - Generasi: {generations}")
    print(f"  - Crossover rate: {crossover_rate:.2f}")
    print(f"  - Mutation rate: {mutation_rate:.2f}")
    print(f"  - CPU Cores: {cpu_cores} (PARALLEL prange mode)")
    
    # for iteration in range(3):
    print(f"\n=== INISIALISASI POPULASI AWAL ===\n")
    # random.seed(42)
    population = []
    
    init_start = time.time()
    
    for i in range(population_size):
        chromosome = create_chromosome(list(packages_dict.values()))
        population.append(chromosome)
        if i < 5 or i >= population_size - 2:
            print(f"K{i+1}: {chromosome[:3]}... ({len(chromosome)} gen)")
        elif i == 5:
            print("  ...")
    
    init_time = time.time() - init_start
    
    print(f"\nPopulasi awal: {population_size} kromosom, masing-masing {len(packages)} gen")
    print(f"Waktu inisialisasi: {init_time:.2f} detik")
    
    print("\n" + "="*60)
    print("MEMULAI ALGORITMA GENETIK (PARALLEL PRANGE MODE)")
    print("="*60)
    
    ga_start_time = time.time()
    current_pop = population.copy()
    all_results = []
    
    # Tracking waktu kumulatif
    total_eval_time = 0
    total_crossover_time = 0
    total_mutation_time = 0
    total_selection_time = 0
    
    for gen in range(generations):
        print(f"\n{'='*60}")
        print(f"GENERASI {gen + 1}")
        print(f"{'='*60}")
        
        # ========== STEP 1: EVALUASI POPULASI ==========
        print("\n>> 1. Evaluasi Populasi (PARALLEL prange):")
        print("   ========================================")
        
        eval_start = time.time()
        fitness_scores, metadata_list = evaluate_population_parallel(current_pop, container, packages_dict)
        eval_time = time.time() - eval_start
        total_eval_time += eval_time
        
        # Tampilkan hasil setiap kromosom
        for i, (chrom, fitness, metadata) in enumerate(zip(current_pop, fitness_scores, metadata_list)):
            all_results.append({
                'Generation': gen + 1,
                'Chromosome': f"K{i+1}",
                'Fitness': fitness,
                'VolumeUtil': metadata['volume_utilization'],
                'B4': metadata['B4'],
                'B5': metadata['B5'],
                'Placed': metadata['num_placed'],
                'Sequence': chrom.copy()
            })
            
            status_b4 = '✓' if metadata['B4'] == 1 else '✗'
            status_b5 = '✓' if metadata['B5'] == 1 else '✗'
            print(f"   K{i+1}: Fitness={fitness:8.2f} | VU={metadata['volume_utilization']:5.2f}% | "
                f"Terpasang={metadata['num_placed']:2}/{len(packages)} | B4={status_b4} B5={status_b5}")
        
        print(f"   ========================================")
        print(f"   Fitness Range: {min(fitness_scores):.2f} - {max(fitness_scores):.2f}")
        print(f"   Fitness Avg: {np.mean(fitness_scores):.2f}")
        print(f"   Waktu Evaluasi: {eval_time:.2f} detik")
        
        best_fitness = max(fitness_scores)
        best_idx = fitness_scores.index(best_fitness)
        best_meta = metadata_list[best_idx]
        
        if gen < generations - 1:
            # ========== STEP 2: CROSSOVER ==========
            print(f"\n>> 2. Membuat Offspring via Crossover:")
            print(f"   ========================================")
            
            crossover_start = time.time()
            target_crossover = int(population_size * crossover_rate)
            offspring_crossover = []
            
            for c in range(target_crossover):
                p1_idx = tournament_selection(current_pop, fitness_scores)
                p2_idx = tournament_selection(current_pop, fitness_scores)
                while p2_idx == p1_idx and len(current_pop) > 1:
                    p2_idx = tournament_selection(current_pop, fitness_scores)
                child = pmx_crossover(current_pop[p1_idx], current_pop[p2_idx])
                offspring_crossover.append(child)
                
                # Print detail crossover
                print(f"   Crossover #{c+1}: Parent K{p1_idx+1} (fit={fitness_scores[p1_idx]:.2f}) + "
                    f"Parent K{p2_idx+1} (fit={fitness_scores[p2_idx]:.2f}) → Child {c+1}")
            
            crossover_time = time.time() - crossover_start
            total_crossover_time += crossover_time
            print(f"   Total crossover: {len(offspring_crossover)} child")
            print(f"   Waktu Crossover: {crossover_time:.2f} detik")
            
            # ========== STEP 3: MUTASI ==========
            print(f"\n>> 3. Membuat Offspring via Mutasi:")
            print(f"   ========================================")
            
            mutation_start = time.time()
            target_mutation = population_size - target_crossover
            offspring_mutation = []
            
            for m in range(target_mutation):
                p_idx = tournament_selection(current_pop, fitness_scores)
                child = mutate(current_pop[p_idx], mutation_rate)
                offspring_mutation.append(child)
                
                # Print detail mutasi
                print(f"   Mutasi #{m+1}: Parent K{p_idx+1} (fit={fitness_scores[p_idx]:.2f}) → Mutant {m+1}")
            
            mutation_time = time.time() - mutation_start
            total_mutation_time += mutation_time
            print(f"   Total mutasi: {len(offspring_mutation)} child")
            print(f"   Waktu Mutasi: {mutation_time:.2f} detik")
            
            # Gabungkan offspring
            offspring = offspring_crossover + offspring_mutation
            print(f"\n   Total offspring: {len(offspring)}")
            
            # ========== STEP 4: EVALUASI OFFSPRING ==========
            print(f"\n>> 4. Evaluasi Offspring (PARALLEL prange):")
            print(f"   ========================================")
            
            offspring_eval_start = time.time()
            offspring_fitness, _ = evaluate_population_parallel(offspring, container, packages_dict)
            offspring_eval_time = time.time() - offspring_eval_start
            total_eval_time += offspring_eval_time
            
            # Print detail evaluasi offspring
            for i, (child, fit) in enumerate(zip(offspring, offspring_fitness)):
                print(f"   Offspring {i+1}: Fitness={fit:8.2f} | {child[:3]}...")
            
            print(f"   Waktu Evaluasi Offspring: {offspring_eval_time:.2f} detik")
            
            # ========== STEP 5: SELEKSI (ELITISM) ==========
            print(f"\n>> 5. Seleksi Elitism (gabung parent + offspring):")
            print(f"   ========================================")
            
            selection_start = time.time()
            combined_pop = current_pop + offspring
            combined_fitness = fitness_scores + offspring_fitness
            sorted_indices = np.argsort(combined_fitness)[::-1]
            current_pop = [combined_pop[i] for i in sorted_indices[:population_size]]
            selection_time = time.time() - selection_start
            total_selection_time += selection_time
            
            print(f"   Total individu sebelum seleksi: {len(combined_pop)}")
            print(f"   Memilih {population_size} terbaik...")
            print(f"   Waktu Seleksi: {selection_time:.2f} detik")
            
            # Tampilkan hasil seleksi
            print(f"\n   >> 10 INDIVIDU TERBAIK DARI SELEKSI:")
            print(f"   ========================================")
            for i, idx in enumerate(sorted_indices[:min(10, population_size)]):
                status = "Parent" if idx < len(fitness_scores) else "Offspring"
                print(f"   Rank {i+1}: Index {idx} ({status}) - Fitness: {combined_fitness[idx]:.2f}")
            print(f"   ========================================")
    
    ga_total_time = time.time() - ga_start_time
    
    # ========== HASIL AKHIR ==========
    print("\n" + "="*60)
    print("HASIL AKHIR")
    print("="*60)
    
    best_result = max(all_results, key=lambda x: x['Fitness'])
    
    print(f"\nSOLUSI TERBAIK:")
    print(f"  Generasi: {best_result['Generation']}")
    print(f"  Kromosom: {best_result['Chromosome']}")
    print(f"  Fitness: {best_result['Fitness']:.2f}")
    print(f"  Volume Utilization: {best_result['VolumeUtil']:.2f}%")
    print(f"  B4 (Kapasitas Beban): {'LULUS' if best_result['B4'] == 1 else 'GAGAL'}")
    print(f"  B5 (Stabilitas): {'LULUS' if best_result['B5'] == 1 else 'GAGAL'}")
    print(f"  Paket Terpasang: {best_result['Placed']}/{len(packages)}")
    
    print(f"\n=== RINGKASAN WAKTU EKSEKUSI ===")
    print(f"  Waktu inisialisasi populasi: {init_time:.2f} detik")
    print(f"  Total waktu evaluasi (populasi + offspring): {total_eval_time:.2f} detik")
    print(f"  Total waktu crossover: {total_crossover_time:.2f} detik")
    print(f"  Total waktu mutasi: {total_mutation_time:.2f} detik")
    print(f"  Total waktu seleksi: {total_selection_time:.2f} detik")
    print(f"  TOTAL WAKTU ALGORITMA GENETIK: {ga_total_time:.2f} detik")
    
    # Visualisasi solusi terbaik
    print("\n" + "="*60)
    print("VISUALISASI SOLUSI TERBAIK")
    print("="*60)
    
    # Untuk visualisasi, evaluasi ulang dengan posisi lengkap
    best_chromosome = best_result['Sequence']
    
    if NUMBA_AVAILABLE:
        n = len(best_chromosome)
        chrom_ids = np.zeros(n, dtype=np.int32)
        chrom_rots = np.zeros(n, dtype=np.int32)
        for i, (pid, rot) in enumerate(best_chromosome):
            chrom_ids[i] = _numba_package_id_to_idx[pid]
            chrom_rots[i] = rot - 1
        
        (fitness, total_volume, total_weight, penalty_cog,
        B1, B2, B3, B4, B5, num_placed,
        cog_total_x, cog_total_y, cog_total_z, dev_x, dev_y, dev_z,
        pos_x, pos_y, pos_z, placed_flags, package_volumes) = bottom_left_fill_numba_optimized(
            chrom_ids, chrom_rots,
            _numba_package_dims, _numba_package_weights,
            container.length, container.width, container.height,
            container.cog_limit_x, container.cog_limit_y, container.cog_limit_z,
            container.max_weight
        )
        
        positions = []
        for i, (pid, rot) in enumerate(best_chromosome):
            dims = packages_dict[pid].orientations[rot]
            if placed_flags[i] == 1:
                positions.append({
                    'id': pid,
                    'x': float(pos_x[i]), 'y': float(pos_y[i]), 'z': float(pos_z[i]),
                    'dx': dims[0], 'dy': dims[1], 'dz': dims[2],
                    'weight': packages_dict[pid].weight,
                    'volume': float(package_volumes[i]),
                    'orientation': rot,
                    'placed': True
                })
            else:
                positions.append({
                    'id': pid,
                    'x': -1, 'y': -1, 'z': -1,
                    'dx': dims[0], 'dy': dims[1], 'dz': dims[2],
                    'weight': packages_dict[pid].weight,
                    'volume': dims[0] * dims[1] * dims[2],
                    'orientation': rot,
                    'placed': False
                })
    else:
        positions = []
    
    if len(positions) > 0:
        fig = visualize_packing(
            positions,
            (container.length, container.width, container.height),
            f"3D Bin Packing - Fitness: {best_result['Fitness']:.2f} | VU: {best_result['VolumeUtil']:.1f}%"
        )
        try:
            fig.show(renderer="browser")
            print("\nVisualisasi 3D dibuka di browser")
            fig.write_html(f"result/visualization.html")
            print(f"\nVisualisasi disimpan ke 'result/visualization.html'")
        except:
            fig.write_html("packing_visualization.html")
            print("\nVisualisasi disimpan ke 'packing_visualization.html'")
            import webbrowser
            webbrowser.open("packing_visualization.html")
    
    # Export hasil ke CSV
    df_export = pd.DataFrame([{
        'Generation': r['Generation'],
        'Chromosome': r['Chromosome'],
        'Fitness': r['Fitness'],
        'VolumeUtilization': r['VolumeUtil'],
        'B4': r['B4'],
        'B5': r['B5'],
        'PlacedPackages': r['Placed']
    } for r in all_results])
    
    df_export.to_csv(f'result/packing_results.csv', index=False)
    print(f"\nHasil lengkap diexport ke 'result/packing_results.csv'")
        
        # population_size += 50
        # generations += 50
        # crossover_rate += 0.05
        # mutation_rate -= 0.05


if __name__ == "__main__":
    main()


>>> MEMBACA FILE dataset.csv <<<

Berhasil memuat 39 paket dari dataset.csv

=== KONTAINER ===
Dimensi: 240.0 x 160.0 x 130.0 cm
Volume: 4992000.0 cm³
Berat maks: 800.0 kg
Batas pusat massa: X=24.0, Y=16.0, Z=19.5 cm

Total volume semua paket: 4456423 cm³
Total berat semua paket: 795.1 kg
Volume kontainer: 4992000.0 cm³
Utilization maksimum teoritis: 89.3%
✅ Numba data initialized for 39 packages

=== KONFIGURASI GENETIC ALGORITHM ===


Konfigurasi GA:
  - Jumlah Paket: 39
  - Populasi: 4
  - Generasi: 3
  - Crossover rate: 0.50
  - Mutation rate: 0.50
  - CPU Cores: 16 (PARALLEL prange mode)

=== INISIALISASI POPULASI AWAL ===

K1: [('P034', 3), ('P016', 4), ('P027', 2)]... (39 gen)
K2: [('P009', 4), ('P028', 6), ('P005', 5)]... (39 gen)
K3: [('P017', 6), ('P022', 2), ('P031', 2)]... (39 gen)
K4: [('P035', 4), ('P024', 3), ('P022', 6)]... (39 gen)

Populasi awal: 4 kromosom, masing-masing 39 gen
Waktu inisialisasi: 0.00 detik

MEMULAI ALGORITMA GENETIK (PARALLEL PRANGE MODE)

GENERAS